# What we own, and what we can actually use

One row per deal. The left half is **what we own** — the book Intrum's own PAM reports:
when we bought it, how many cases, how much. The right half is **what we can use** — what
has actually arrived in a feed we can query, and how far it reaches.

The query is [`deal_data_coverage.sql`](deal_data_coverage.sql); every guard and why it is
there lives in its header. This notebook runs it, adds the `notes` column explaining each
gap, and exports the table.

| Column | Means |
|---|---|
| `deal_id` | Heimdall deals under Intrum's id, every other vehicle under its Atlas id |
| `capex_amount` | PAM `acquisitionprincipal` — the face value acquired |
| `purchase_price` | PAM `rp_purchase_price` — what was paid. The dims call *this* capex |
| `cat_*` | The raw Intrum CAT feed: the servicer's own case and payment records |
| `pam_*` | Intrum's PAM reporting: the book as Intrum accounts for it |
| `in_asset_index`, `bridge_account_coverage_pct` | Deliberately empty — filled by hand for now |

⚠ **Collections are not currency-converted.** `prod_collections_eur` and `pam_collections` are
each in the deal's own currency, so the columns must never be summed down the page. Deals
with more than one currency are flagged by `pam_currencies > 1`.


In [ ]:
import sys
sys.path.insert(0, "../../packages")   # notebook sits in work/workshop/

import datalake
import pandas as pd

# 5 dp per the workspace convention: this changes only what is PRINTED.
pd.set_option("display.float_format", lambda v: f"{v:,.5f}")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)


## 1. Load

`convert=False` keeps Snowflake's `Decimal` values instead of float64, because these are
money columns that get summed and reconciled.


In [ ]:
# Reads the whole CAT transaction feed, so it is not quick.
deals = datalake.q("work/workshop/deal_data_coverage", convert=False)

# convert=False leaves dates as objects; coerce the four we chart or sort on.
for col in ("CAPEX_DATE", "PROD_LATEST_PAYMENT", "PAM_LATEST_MONTH"):
    deals[col] = pd.to_datetime(deals[col], errors="coerce")

print(f"{len(deals):,} deals")
deals.head(10)


## 2. Why a deal is thin

`notes` is the column the slide is actually built on: it says, in words, why a deal has
less data than the book says it should. Most of it is derived, so it stays true when the
query is re-run. `KNOWN` below is the hand-written part — reasons no column can show,
like a feed that exists but has not been modelled yet. Add to it as you learn things.


In [ ]:
# Reasons that are not visible in any column. Keyed on deal_id; edit freely.
KNOWN = {
    # "SOUQ":    "Feed arrives, but no dbt model yet -- nothing queryable downstream.",
}


def note(row):
    """Why this deal is thinner than the book says, most pressing reason first."""
    said = []
    if row.deal_id in KNOWN:
        said.append(KNOWN[row.deal_id])

    # Owned but no servicer feed at all: the whole right-hand side is empty.
    if pd.isna(row.pam_cases_acquired) and pd.isna(row.prod_cases_with_a_payment):
        said.append("In neither Intrum feed -- not an Intrum-serviced deal, or not yet delivered.")
    else:
        if pd.isna(row.prod_cases_with_a_payment):
            said.append("No cases in the CAT feed, so nothing is queryable at account level.")
        if pd.isna(row.pam_cases_acquired):
            said.append("Not in the pinned PAM file -- no book figures to compare against.")

    # Both feeds present but they disagree on how much has been collected.
    if row.prod_vs_pam_pct is not None and not pd.isna(row.prod_vs_pam_pct):
        if row.prod_vs_pam_pct < 90:
            said.append(f"CAT holds only {row.prod_vs_pam_pct:.1f}% of what PAM reports collected.")
        elif row.prod_vs_pam_pct > 110:
            said.append(f"CAT holds {row.prod_vs_pam_pct:.1f}% of PAM -- more than the book: check the instance.")

    # Undated payments never reach a time series, however complete the total is.
    if row.prod_cases_pct is not None and not pd.isna(row.prod_cases_pct) and row.prod_cases_pct >= 5:
        said.append(f"{row.prod_cases_pct:.1f}% of CAT payments carry no date -- in the total, absent from any month.")

    # A feed that stopped: the book is still running but nothing new has arrived.
    if not pd.isna(row.prod_months_behind_pam) and row.prod_months_behind_pam >= 3:
        said.append(f"CAT is {int(row.prod_months_behind_pam)} months behind PAM -- the feed has gone quiet.")

    if row.pam_currencies is not None and not pd.isna(row.pam_currencies) and row.pam_currencies > 1:
        said.append(f"{int(row.pam_currencies)} currencies on one deal -- its totals must not be added up.")

    return " ".join(said)


### The derived comparisons `note()` reads


In [ ]:
# Share of PAM's reported collections that the CAT feed can actually account for.
# Both are in the deal's own currency, so the ratio is safe even though the totals
# are not comparable between deals.
deals["prod_vs_pam_pct"] = (
    deals["PROD_COLLECTIONS_EUR_TO_PAM_MONTH"].astype("float64")
    / deals["PAM_COLLECTIONS_EUR"].astype("float64").replace(0, pd.NA) * 100
)

# Payments with no transaction_date: in the total, but invisible to any month.
deals["prod_cases_pct"] = (
    deals["PROD_CASES_WITH_A_PAYMENT"].astype("float64")
    / deals["PROD_CASES_WITH_A_PAYMENT"].astype("float64").replace(0, pd.NA) * 100
)

# How far the servicer feed lags the book.
deals["prod_months_behind_pam"] = (
    (deals["PAM_LATEST_MONTH"] - deals["PROD_LATEST_PAYMENT"]).dt.days / 30.44
).round(0)

# note() reads lower-case attributes, so give it a lower-cased view.
lower = deals.rename(columns=str.lower)
deals["notes"] = [note(r) for r in lower.itertuples(index=False)]

print(f"{(deals['notes'] != '').sum():,} of {len(deals):,} deals have something worth saying")


## 3. The two halves

`owned` is every deal with a book figure. `usable` is the subset whose account-level data
has arrived and reconciles — the honest denominator for anything built on loan-level data.


In [ ]:
owned = deals["CAPEX_AMOUNT_EUR"].notna()
usable = owned & deals["PROD_CASES_WITH_A_PAYMENT"].notna() & (deals["prod_vs_pam_pct"] >= 90)

summary = pd.DataFrame({
    "deals": [owned.sum(), usable.sum()],
    "cases": [deals.loc[owned, "CASES_ACQUIRED"].astype("float64").sum(),
              deals.loc[usable, "CASES_ACQUIRED"].astype("float64").sum()],
}, index=["What we own", "What we can use"])
summary["% of deals"] = 100 * summary["deals"] / summary.loc["What we own", "deals"]
summary["% of cases"] = 100 * summary["cases"] / summary.loc["What we own", "cases"]
summary


## 4. The table


In [ ]:
COLUMNS = [
    "deal_id", "vehicle", "country", "seller_name", "capex_date", "currency",
    "cases_acquired", "capex_amount_local", "capex_amount_eur",
    "purchase_price_local", "purchase_price_eur", "rate_to_eur",
    "in_asset_index", "bridge_account_coverage_pct",
    "prod_months", "pam_months",
    "prod_collections_local", "pam_collections_local",
    "prod_collections_eur", "pam_collections_eur",
    "prod_collections_eur_to_pam_month",
    "prod_latest_payment", "pam_latest_month", "notes",
]
table = deals.rename(columns=str.lower)[COLUMNS]
table.to_csv("deal_data_coverage.csv", index=False)
print(f"wrote deal_data_coverage.csv -- {len(table):,} rows")
table.head(25)


---
## 5. What the raw feed holds, and what reached PROD

The deal table above only knows what dbt models. This section is the ledger behind that:
every table Intrum actually sends, and whether anything downstream can read it.

`IN_DBT` comes from the declared sources in
`brocc-data/transformation/dbt/models/staging/intrum_cat/source.yml`. Re-check it there
rather than trusting this list — it has already changed once: `settlement`, `installment`
and `group_case_phase` were all unmodelled in the September 2026 review and are sources now.


In [ ]:
# The seven tables dbt declares as sources, out of the 37 the feed delivers.
# From brocc-data .../staging/intrum_cat/source.yml.
DBT_SOURCES = {
    "CASES", "DEBTORS", "GROUP_CASE_PHASE", "INSTALLMENT",
    "INVOICES", "SETTLEMENT", "TRANSACTIONS",
}

# Where a modelled source actually surfaces, and what is dropped on the way.
REACHES = {
    "CASES":            "DIM_CASE, FCT_CASE_SNAPSHOT_DAILY -- status only, no legal field",
    "TRANSACTIONS":     "FCT_CASE_COLLECTION",
    "DEBTORS":          "DIM_BORROWER, BRG_CASE_BORROWER",
    "INVOICES":         "int_npl_invoices_grouped",
    "GROUP_CASE_PHASE": "MART_NPL_SERVICING_AGG_CASES_MONTHLY only -- aggregated, no case-level phase",
    "INSTALLMENT":      "FCT_CASE_FORBEARANCE",
    "SETTLEMENT":       "FCT_CASE_FORBEARANCE",
}

raw_tables = datalake.q("""
    SELECT table_name, row_count
    FROM raw.information_schema.tables
    WHERE table_schema = 'HYPERION_INTRUM_CAT'
      AND table_name NOT LIKE '\\_DLT%'
    ORDER BY row_count DESC
""")

raw_tables["in_dbt"] = raw_tables["TABLE_NAME"].isin(DBT_SOURCES)
raw_tables["reaches"] = raw_tables["TABLE_NAME"].map(REACHES).fillna("-- nothing downstream reads it --")

print(f"{raw_tables['in_dbt'].sum()} of {len(raw_tables)} raw tables are dbt sources")
print(f"{raw_tables.loc[~raw_tables['in_dbt'], 'ROW_COUNT'].sum():,.0f} rows sit in tables nothing reads")
raw_tables


### The legal and status tables specifically

These are the ones that would answer a question about proceedings. None is a dbt source.


In [ ]:
LEGAL_TABLES = [
    ("LEGAL_STEPS",                "the legal track, step by step"),
    ("LEGAL_INFO",                 "legal detail per case"),
    ("LEGAL_RESULT_INFO",          "the outcome of a legal step"),
    ("LEGAL_STEP_PARTIES",         "who is party to each step"),
    ("BANKRUPTCY_RECORD",          "insolvency / bankruptcy track"),
    ("CASE_STATUS_CHANGE_HISTORY", "status over time, not just the latest"),
    ("INSTALLMENT_PLAN_ITEM",      "the instalments inside a payment plan"),
    ("ASSIGNMENT",                 "case assignment to a collector or agency"),
]
legal = raw_tables[raw_tables.TABLE_NAME.isin([t for t, _ in LEGAL_TABLES])].copy()
legal["holds"] = legal.TABLE_NAME.map(dict(LEGAL_TABLES))
legal[["TABLE_NAME", "ROW_COUNT", "holds", "in_dbt", "reaches"]]


---
## 6. Country-specific legal information: do we hold it, and is it cleaned?

Each country runs its own enforcement process, so each needs a different field to answer
"where is this case legally?". The table below is one row per **(country, legal question)**:
which raw field would carry the answer, what share of that country's cases actually fill it,
and whether dbt cleans it.

**The dbt answer is the same for every row: no.** `DIM_CASE` carries `STATUS_CODE`,
`STATUS_DATE` and `STATUS_DESCRIPTION` and nothing else at case level — no phase, no
limitation date, no judgement, no court field. Every `LEGAL_*` column in PROD is
`LEGAL_BUYER_SK`, which is who owns the debt, not any proceeding.

⚠ Fill shares are `APPROX_COUNT_DISTINCT` over a 6.5bn-row change history — good enough
for yes/no, not a figure to quote to two decimals.


In [ ]:
fill = datalake.q("work/workshop/legal_fields_by_country")
print(f"{len(fill):,} (country, field) pairs across {fill.COUNTRY.nunique()} countries")

# The legal question each country actually needs answered, and the raw field that would
# carry it. Edit as you learn more -- this is judgement, not something the data states.
COUNTRY_LEGAL = {
    "FI": [("Limitation / statute-barred date (velan vanhentuminen)", "statute_barred_date (limitation date)"),
           ("Enforcement by ulosotto", "debt_enforcement_number (bailiff ref)"),
           ("Enforceable title held", "is_titled (enforceable title held)")],
    "SE": [("Enforcement by Kronofogden / utslag", "debt_enforcement_number (bailiff ref)"),
           ("Judgement obtained", "judgement_date"),
           ("Limitation (preskription)", "statute_barred_date (limitation date)")],
    "FR": [("Titre executoire / judgement", "judgement_date"),
           ("Judgement obtained", "judgement_date")],
    "DK": [("Enforcement (fogedret)", "debt_enforcement_number (bailiff ref)"),
           ("Judgement obtained", "judgement_date")],
    "DE": [("Titel held (30-year enforceability)", "is_titled (enforceable title held)"),
           ("Mahnbescheid / judgement", "judgement_date"),
           ("Limitation (Verjaehrung)", "statute_barred_date (limitation date)")],
    "CH": [("Betreibung register reference", "debt_enforcement_number (bailiff ref)"),
           ("Judgement / Rechtsoeffnung", "judgement_date")],
    "PT": [("Injuncao / executive proceeding", "court_procedure"),
           ("Phase (amicable vs legal)", "case_phase (amicable/legal/execution stage)")],
    "IT": [("Decreto ingiuntivo / pignoramento", "court_procedure"),
           ("Legal flag carried on the product", "product_id (the legal flag in Italy)"),
           ("Insolvency / restructuring", "restructuring_date (insolvency/plan)")],
    "ES": [("Monitorio / ejecutivo", "court_procedure"),
           ("Phase (amicable vs legal)", "case_phase (amicable/legal/execution stage)")],
    "PL": [("Nakaz zaplaty / komornik", "debt_enforcement_number (bailiff ref)"),
           ("Judgement obtained", "judgement_date")],
    "GR": [("Diatagi pliromis (payment order)", "court_procedure"),
           ("Phase (amicable vs legal)", "case_phase (amicable/legal/execution stage)")],
}

# Instruments, which is where promissory notes live. Relevant everywhere.
EVERYWHERE = [("Instrument type (promissory note, invoice, loan)", "receivable_type (promissory note, invoice, loan)"),
              ("Collection status", "main_status_code (collection status)"),
              ("Why the case closed", "closing_code (why it closed)"),
              ("Insolvency / restructuring", "restructuring_date (insolvency/plan)")]


In [ ]:
rows = []
for country, items in COUNTRY_LEGAL.items():
    for question, field in items + EVERYWHERE:
        hit = fill[(fill.COUNTRY == country) & (fill.FIELD == field)]
        if hit.empty:
            rows.append((country, question, field, None, "no cases in the feed"))
            continue
        pct = float(hit.PCT_OF_CASES.iloc[0])
        held = "yes" if pct >= 50 else ("partly" if pct > 0 else "NO -- column empty")
        rows.append((country, question, field, pct, held))

legal_by_country = pd.DataFrame(
    rows, columns=["country", "legal question", "raw field", "pct_of_cases", "in the database?"]
).drop_duplicates(subset=["country", "legal question"])

# One answer for every row, and it is the point of the section.
legal_by_country["cleaned in dbt?"] = "NO -- raw only"

legal_by_country.to_csv("legal_coverage_by_country.csv", index=False)
print(f"wrote legal_coverage_by_country.csv -- {len(legal_by_country)} rows")
legal_by_country.sort_values(["country", "pct_of_cases"], ascending=[True, False])


### Does it vary by servicer?

Yes, completely — and not in a way this table can show, because **all of it is Intrum**.
The CAT feed is Intrum's. The other servicers send either no case feed at all (Cerved,
Gescobro, Fire) or cases with no phase on them (Massey, Souq). So for every non-Intrum
book the answer to each row above is "we hold nothing", whatever the country.


### What the scan found, measured 2026-11-02

Read the catalogue above against this, not the other way round — several of my
country-to-field guesses are wrong, and the measurement is what is true.

| Field | FI | SE | DK | CH | PT | IT | ES | PL | GR |
|---|---|---|---|---|---|---|---|---|---|
| `main_status_code` | 100 | 100 | 100 | 100 | 100 | 100 | 100 | 100 | 100 |
| `product_id` | 99 | 100 | 100 | 100 | 100 | 100 | 100 | 100 | 100 |
| `case_phase` | **0** | **9** | 100 | 100 | 77 | 100 | 100 | 100 | 100 |
| `statute_barred_date` (limitation) | **6** | **9** | 100 | 54 | 0 | 0 | 0 | 85 | 0 |
| `judgement_date` | 42 | 36 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| `is_titled` | 0 | 0 | 0 | 0 | 0 | 100 | 0 | 100 | 0 |
| `receivable_type` | 0 | 0 | 100 | 0 | 0 | 100 | 100 | 0 | 0 |
| `debt_enforcement_number` | 0 | 0 | 0 | 30 | 0 | 0 | 0 | 0 | 0 |
| `court_procedure` | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| `statutory_date` | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |

Four things stand out:

- **`court_procedure` and `statutory_date` are empty in every country.** The columns exist
  and nobody fills them, so no proceeding type can be read from the case feed at all.
- **Finland's limitation date is 6% filled** — the one field the question was really about.
  It is near-complete in Denmark (100%) and Poland (85%) instead.
- **Finland has no phase at all (0%), Sweden almost none (9%)** while every southern book is
  at 100%. The amicable/legal split simply does not exist for the Nordics.
- **`debt_enforcement_number` is empty for Sweden and Finland** — so Kronofogden and ulosotto
  references are not in the field whose name suggests them. They may be in `client_ref_*`
  or not sent at all; worth one question to Intrum.
